## LLM для кода: bug fixing + LoRA fine-tuning


#### Карта задач NLP для кода (NLP for Code Tasks Map)

Удобно рассматривать задачи NLP для кода как несколько уровней.

##### 1) Понимание кода (Code Understanding)

* суммаризация кода (code summarization);
* генерация docstring (docstring generation);
* именование переменных и функций (naming);
* поиск кода (code search / retrieval);
* эмбеддинги для поиска похожих фрагментов (embeddings for similarity search).

##### 2) Преобразование кода (Code Transformation)

* исправление ошибок (bug fixing);
* рефакторинг (refactoring);
* оптимизация (optimization);
* приведение к стилю (style transfer);
* перевод между языками программирования (code translation between programming languages).

##### 3) Генерация (Code Generation)

* генерация кода по текстовому описанию (text-to-code);
* автодополнение (autocomplete);
* дописывание функций (function completion);
* заполнение пропусков в коде (infilling).

##### 4) Инструменты разработчика (Developer Tools)

* генерация тестов (test generation);
* ревью кода (code review);
* объяснение ошибок (error explanation);
* поиск уязвимостей (security scanning);
* помощь в написании документации (documentation assistance).

##### 5) Системный уровень (System-Level Applications)

* интеллектуальные ассистенты для программирования (code assistants);
* RAG по репозиторию (repository-level RAG);
* агентные системы и многошаговые пайплайны исправления кода (agents / multi-step repair pipelines).

#####

Чем задача ближе к преобразованию кода (code transformation) и чем более четко определены ее вход и выход (well-defined input/output), тем проще получить заметный эффект от fine-tuning. Он обычно помогает тогда, когда модель нужно подстроить под конкретный формат преобразования, а не научить “в целом хорошо программировать”.



- [A Survey of Machine Learning for Big Code and Naturalness (Allamanis et al., 2018)](https://arxiv.org/abs/1709.06182)  
  Классическая работа, вводит понятие *naturalness of code* и описывает базовые задачи: автодополнение, суммаризация, именование, поиск кода.

- [Pre-trained Models for Code: A Survey](https://arxiv.org/abs/2103.07950)  
  Обзор предобученных моделей (CodeBERT, CodeT5 и др.) и структурирование задач: понимание, генерация, трансформация кода.

- [Large Language Models for Code: A Survey](https://arxiv.org/abs/2310.12148)  
  Современный обзор LLM для кода: генерация, reasoning, оценка качества (например, HumanEval), ограничения и направления развития.

#### bug fixing

- вход и выход похожи;
- правка обычно локальная;
- можно сделать synthetic dataset;
- можно сравнить результат до и после обучения;
- проще объяснить, что именно модель должна выучить.

Пример:

```python
def mean(arr):
    return sum(arr) / len(arr) - 1
```

Ожидаемый фикс:

```python
def mean(arr):
    return sum(arr) / len(arr)
```


#### CONFIG: все основные настройки в одном месте


In [ ]:
# =========================
# CONFIG
# =========================

CONFIG = {
    # API baseline через DeepSeek
    "use_deepseek_api": True,

    # Локальная модель
    "use_local_model": True,

    # Вариант 1: модель скачана вручную в Windows
    "local_model_name": "/mnt/c/models/deepseek-coder-1.3b-instruct",

    # Вариант 2: модель из HF, если сеть работает
    # "local_model_name": "deepseek-ai/deepseek-coder-1.3b-instruct",

    # Папка для LoRA-адаптера
    "output_dir": "./bugfix_lora_adapter",

    # Генерация
    "max_new_tokens": 160,
    "temperature": 0.0,

    # Обучение
    "max_length": 256,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,

    # Управление запуском обучения
    # Для семинара безопаснее False, если адаптер уже обучен заранее
    "run_training": True,

    "seed": 42,
}


#### Переменные окружения HuggingFace и импорты


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Эти переменные должны быть установлены ДО импорта transformers / huggingface_hub.
# Если модель уже скачана локально в C:\models, они не критичны, но не мешают.
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# os.environ["HF_HUB_DISABLE_XET"] = "1"
# os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
# os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

# print("HF_ENDPOINT =", os.environ.get("HF_ENDPOINT"))
# print("HF_HUB_DISABLE_XET =", os.environ.get("HF_HUB_DISABLE_XET"))
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL = "deepseek-chat"


In [ ]:
import re
import json
import random
from typing import Optional, Dict, Any, Callable, List

import torch
import pandas as pd

from IPython.display import display, Markdown

random.seed(CONFIG["seed"])


#### Utility-функции: нормализация, красивый вывод, метрики


In [ ]:
def normalize_code(code: str) -> str:
    code = code.strip()
    code = code.replace("```python", "").replace("```", "")
    return "\n".join(line.rstrip() for line in code.splitlines()).strip()


def exact_match_score(pred: str, gold: str) -> int:
    pred_norm = normalize_code(pred)
    gold_norm = normalize_code(gold)
    return int(pred_norm == gold_norm)


def contains_gold_score(pred: str, gold: str) -> int:
    return int(normalize_code(gold) in normalize_code(pred))


def pretty_print_result(result: Dict[str, Any], title: str = "Model result"):
    """Красивый вывод JSON-ответа модели."""
    code_text = result.get("code", "")
    explanation = result.get("explanation", "")

    display(Markdown(f"### {title}"))
    display(Markdown(f"**Fixed code:**\n```python\n{code_text}\n```"))
    display(Markdown(f"**Explanation:**\n\n{explanation}"))


def pretty_print_sample(sample: dict, title: str = "Bug-fixing sample"):
    display(Markdown(f"### {title}"))
    display(Markdown(f"**Name:** `{sample.get('name', '')}`"))
    display(Markdown(f"**Bug type:** `{sample.get('bug_type', '')}`"))

    display(Markdown(
        "**Buggy code:**\n"
        f"```python\n{sample['buggy_code']}\n```"
    ))

    display(Markdown(
        "**Expected fixed code:**\n"
        f"```python\n{sample['fixed_code']}\n```"
    ))


def show_eval_table(rows: List[Dict[str, Any]]):
    """Показывает таблицу результатов evaluation."""
    df = pd.DataFrame(rows)
    display(df)
    if "match" in df.columns and len(df) > 0:
        print("Exact match:", round(df["match"].mean(), 3))
    return df


#### Prompt-функции


In [ ]:
def make_bugfix_prompt(sample: Dict[str, Any]) -> str:
    """Создаёт prompt для bug fixing из sample."""
    return f"""
Fix the bug in this Python code.

Return:
1. corrected code
2. short explanation

Code:
{sample["buggy_code"]}
""".strip()


def make_local_bugfix_prompt(sample: Dict[str, Any]) -> str:
    """Более строгий prompt для локальной модели."""
    return f"""
Fix the bug in this Python code. Return only corrected code.

Code:
{sample["buggy_code"]}

Corrected code:
""".strip()


#### DeepSeek API baseline


In [ ]:
import os
from openai import OpenAI


def deepseek_generate(prompt: str) -> Optional[Dict[str, Any]]:
    """Вызывает DeepSeek API и возвращает dict с code и explanation."""
    if not CONFIG["use_deepseek_api"]:
        print("DeepSeek API is disabled")
        return None

    api_key = os.environ.get("DEEPSEEK_API_KEY")
    if not api_key:
        raise RuntimeError(
            "DEEPSEEK_API_KEY не найден. "
            "Задай его в окружении или отключи CONFIG['use_deepseek_api']."
        )

    client = OpenAI(
        api_key=api_key,
        base_url="https://api.deepseek.com",
    )

    system_prompt = """
You are a careful programming assistant.

When asked to fix code, return only valid JSON with exactly two fields:
{
  "code": "corrected Python code as a string",
  "explanation": "short explanation of the bug and the fix"
}

Do not return Markdown.
Do not use ``` code blocks.
Do not add any text outside JSON.
""".strip()

    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        temperature=CONFIG["temperature"],
        stream=False,
        response_format={"type": "json_object"},
    )

    content = response.choices[0].message.content
    if not content or not content.strip():
        raise ValueError("Empty response from DeepSeek API.")

    return json.loads(content)


#### Примеры


In [ ]:
DEMO_SAMPLES = [
    {
        "name": "Arithmetic bug",
        "bug_type": "operator",
        "buggy_code": "def add(a, b):\n    return a - b",
        "fixed_code": "def add(a, b):\n    return a + b",
    },
    {
        "name": "Mean bug",
        "bug_type": "extra_constant",
        "buggy_code": "def mean(arr):\n    return sum(arr) / len(arr) - 1",
        "fixed_code": "def mean(arr):\n    return sum(arr) / len(arr)",
    },
    {
        "name": "Off-by-one",
        "bug_type": "indexing",
        "buggy_code": "def last_index(items):\n    return len(items)",
        "fixed_code": "def last_index(items):\n    return len(items) - 1",
    },
    {
        "name": "Comparison bug",
        "bug_type": "condition",
        "buggy_code": "def is_even(n):\n    return n % 2 == 1",
        "fixed_code": "def is_even(n):\n    return n % 2 == 0",
    },
]

pd.DataFrame(DEMO_SAMPLES)


,name,bug_type,buggy_code,fixed_code
0,Arithmetic bug,operator,"def add(a, b):\n return a - b","def add(a, b):\n return a + b"
1,Mean bug,extra_constant,def mean(arr):\n return sum(arr) / len(arr)...,def mean(arr):\n return sum(arr) / len(arr)
2,Off-by-one,indexing,def last_index(items):\n return len(items),def last_index(items):\n return len(items) - 1
3,Comparison bug,condition,def is_even(n):\n return n % 2 == 1,def is_even(n):\n return n % 2 == 0


In [ ]:
EVAL_SAMPLES = [
    {
        "name": "median_wrong_index",
        "bug_type": "off_by_one_index",
        "buggy_code": """def median_sorted(values):
    middle = len(values) // 2
    return values[middle + 1]""",
        "fixed_code": """def median_sorted(values):
    middle = len(values) // 2
    return values[middle]""",
    },
    {
        "name": "specificity_wrong_denominator",
        "bug_type": "wrong_denominator",
        "buggy_code": """def specificity(tn, fp):
    return tn / (tn + tn)""",
        "fixed_code": """def specificity(tn, fp):
    return tn / (tn + fp)""",
    },
    {
        "name": "negative_filter_wrong_sign",
        "bug_type": "wrong_condition",
        "buggy_code": """def keep_negative(values):
    result = []
    for x in values:
        if x > 0:
            result.append(x)
    return result""",
        "fixed_code": """def keep_negative(values):
    result = []
    for x in values:
        if x < 0:
            result.append(x)
    return result""",
    },
    {
        "name": "min_max_scale_wrong_formula",
        "bug_type": "wrong_formula",
        "buggy_code": """def min_max_scale(values):
    low = min(values)
    high = max(values)
    return [(x - high) / (high - low) for x in values]""",
        "fixed_code": """def min_max_scale(values):
    low = min(values)
    high = max(values)
    return [(x - low) / (high - low) for x in values]""",
    },
    {
        "name": "deduplicate_wrong_membership",
        "bug_type": "wrong_membership",
        "buggy_code": """def unique_values(values):
    seen = set()
    result = []
    for value in values:
        if value in seen:
            result.append(value)
        seen.add(value)
    return result""",
        "fixed_code": """def unique_values(values):
    seen = set()
    result = []
    for value in values:
        if value not in seen:
            result.append(value)
        seen.add(value)
    return result""",
    },
    {
        "name": "count_words_wrong_increment",
        "bug_type": "wrong_increment",
        "buggy_code": """def count_words(words):
    counts = {}
    for word in words:
        counts[word] = counts.get(word, 0)
    return counts""",
        "fixed_code": """def count_words(words):
    counts = {}
    for word in words:
        counts[word] = counts.get(word, 0) + 1
    return counts""",
    },
]

pd.DataFrame(EVAL_SAMPLES)

,name,bug_type,buggy_code,fixed_code
0,median_wrong_index,off_by_one_index,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...
1,specificity_wrong_denominator,wrong_denominator,"def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ..."
2,negative_filter_wrong_sign,wrong_condition,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...
3,min_max_scale_wrong_formula,wrong_formula,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...
4,deduplicate_wrong_membership,wrong_membership,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...
5,count_words_wrong_increment,wrong_increment,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...


#### API baseline

In [ ]:
sample = DEMO_SAMPLES[1]
prompt = make_bugfix_prompt(sample)
api_result = deepseek_generate(prompt)
pretty_print_result(api_result, title="DeepSeek API baseline")


### DeepSeek API baseline

**Fixed code:**
```python
def mean(arr):
    return sum(arr) / len(arr)
```

**Explanation:**

The original code subtracted 1 from the mean, which is incorrect. The mean is simply the sum divided by the count.

In [ ]:
sample = EVAL_SAMPLES[0]
prompt = make_bugfix_prompt(sample)
api_result = deepseek_generate(prompt)
pretty_print_sample(sample, title="Input example")
pretty_print_result(api_result, title="DeepSeek API baseline")


### Input example

**Name:** `median_wrong_index`

**Bug type:** `off_by_one_index`

**Buggy code:**
```python
def median_sorted(values):
    middle = len(values) // 2
    return values[middle + 1]
```

**Expected fixed code:**
```python
def median_sorted(values):
    middle = len(values) // 2
    return values[middle]
```

### DeepSeek API baseline

**Fixed code:**
```python
def median_sorted(values):
    if not values:
        return None
    n = len(values)
    if n % 2 == 1:
        return values[n // 2]
    else:
        mid = n // 2
        return (values[mid - 1] + values[mid]) / 2.0
```

**Explanation:**

The original code incorrectly used `middle + 1` as index, which returns an element after the median for odd-length lists and fails to average the two middle elements for even-length lists. The fix correctly handles both cases: for odd length, return the middle element; for even length, return the average of the two middle elements.

#### Локальная модель: загрузка


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM


tokenizer = None
base_model = None


def load_local_model():
    """Загружает локальную модель и токенизатор."""
    global tokenizer, base_model

    if not CONFIG["use_local_model"]:
        print("Local model is disabled")
        return None, None

    model_name = CONFIG["local_model_name"]
    print("Loading local model from:", model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )

    print("Local model loaded")
    return base_model, tokenizer


In [ ]:
base_model, tokenizer = load_local_model()


Loading local model from: /mnt/c/models/deepseek-coder-1.3b-instruct
Local model loaded


#### Генерация локальнй моделью


In [ ]:
def local_generate(prompt: str, model=None, max_new_tokens: int = None) -> str:
    """Генерация локальной модели. Возвращает только completion, без prompt."""
    current_model = model if model is not None else base_model
    if current_model is None:
        raise RuntimeError("Model is not loaded")

    max_new_tokens = max_new_tokens or CONFIG["max_new_tokens"]

    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to(current_model.device) for k, v in inputs.items()}

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = current_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][input_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


#### Local baseline



In [ ]:
sample = DEMO_SAMPLES[1]
prompt = make_local_bugfix_prompt(sample)

print("BUGGY:")
print(sample["buggy_code"])

print("\nGOLD:")
print(sample["fixed_code"])

print("\nLOCAL BASELINE:")
print(local_generate(prompt, model=base_model))


BUGGY:
def mean(arr):
    return sum(arr) / len(arr) - 1

GOLD:
def mean(arr):
    return sum(arr) / len(arr)

LOCAL BASELINE:
def mean(arr):
    return sum(arr) / len(arr)

Explanation:
The original code was subtracting 1 from the sum of the array elements divided by the length of the array. This is not necessary and can be removed. The corrected code simply returns the sum of the array divided by its length.


In [ ]:
sample = EVAL_SAMPLES[1]
prompt = make_local_bugfix_prompt(sample)

print("BUGGY:")
print(sample["buggy_code"])

print("\nGOLD:")
print(sample["fixed_code"])

print("\nLOCAL BASELINE:")
print(local_generate(prompt, model=base_model))

BUGGY:
def specificity(tn, fp):
    return tn / (tn + tn)

GOLD:
def specificity(tn, fp):
    return tn / (tn + fp)

LOCAL BASELINE:
def specificity(tn, fp):
    return tn / (tn + fp)

Note: The function specificity(tn, fp) should return the specificity of the test, not the false positive rate (fp).

Also, the function specificity(tn, fp) should return the specificity of the test, not the false positive rate (fp).

The function specificity(tn, fp) should return the specificity of the test, not the false positive rate (fp).

The function specificity(tn, fp) should return the specificity of the test, not the false positive rate (fp).

The function specificity(tn, fp) should return the specificity of


#### Evaluation-функция для локальной модели


In [ ]:
def evaluate_local_model(samples: List[Dict[str, Any]], model, title: str = "model") -> List[Dict[str, Any]]:
    """Оценивает модель на наборе samples."""
    rows = []

    for sample in samples:
        prompt = make_local_bugfix_prompt(sample)
        prediction = local_generate(prompt, model=model)
        match = exact_match_score(prediction, sample["fixed_code"])

        rows.append({
            "name": sample["name"],
            "bug_type": sample["bug_type"],
            "buggy_code": sample["buggy_code"],
            "gold": sample["fixed_code"],
            f"{title}_prediction": prediction,
            "match": match,
        })

    return rows


#### Baseline evaluation


In [ ]:
baseline_rows = evaluate_local_model(
    samples=EVAL_SAMPLES,
    model=base_model,
    title="base",
)

baseline_df = show_eval_table(baseline_rows)


,name,bug_type,buggy_code,gold,base_prediction,match
0,median_wrong_index,off_by_one_index,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,0
1,specificity_wrong_denominator,wrong_denominator,"def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...",0
2,negative_filter_wrong_sign,wrong_condition,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,0
3,min_max_scale_wrong_formula,wrong_formula,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,0
4,deduplicate_wrong_membership,wrong_membership,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,0
5,count_words_wrong_increment,wrong_increment,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,0


Exact match: 0.0


In [ ]:
def clean_prediction(text: str) -> str:
    text = str(text).replace("```python", "").replace("```", "").strip()

    for marker in [
        "###",
        "Fix the bug",
        "Code:",
        "Corrected code:",
        "Explanation:",
    ]:
        pos = text.find(marker)
        if pos != -1:
            text = text[:pos].strip()

    return text.strip()

In [ ]:
baseline_df["base_prediction_clean"] = baseline_df["base_prediction"].apply(clean_prediction)

baseline_df["exact_match_clean"] = baseline_df.apply(
    lambda row: exact_match_score(row["base_prediction_clean"], row["gold"]),
    axis=1,
)

baseline_df["contains_gold_clean"] = baseline_df.apply(
    lambda row: contains_gold_score(row["base_prediction_clean"], row["gold"]),
    axis=1,
)

display(baseline_df)

print("Raw exact match:", round(baseline_df["match"].mean(), 3))
print("Clean exact match:", round(baseline_df["exact_match_clean"].mean(), 3))
print("Clean contains gold:", round(baseline_df["contains_gold_clean"].mean(), 3))

,name,bug_type,buggy_code,gold,base_prediction,match,base_prediction_clean,exact_match_clean,contains_gold_clean
0,median_wrong_index,off_by_one_index,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,0,def median_sorted(values):\n middle = len(v...,1,1
1,specificity_wrong_denominator,wrong_denominator,"def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...",0,"def specificity(tn, fp):\n return tn / (tn ...",0,1
2,negative_filter_wrong_sign,wrong_condition,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,0,def keep_negative(values):\n result = []\n ...,0,1
3,min_max_scale_wrong_formula,wrong_formula,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,0,def min_max_scale(values):\n low = min(valu...,1,1
4,deduplicate_wrong_membership,wrong_membership,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,0,def unique_values(values):\n seen = set()\n...,0,0
5,count_words_wrong_increment,wrong_increment,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,0,def count_words(words):\n counts = {}\n ...,0,0


Raw exact match: 0.0
Clean exact match: 0.333
Clean contains gold: 0.667


#### Synthetic dataset: строим обучающие пары buggy → fixed


In [ ]:
import random
from typing import List, Dict, Any


def build_harder_synthetic_samples(copies: int = 8, seed: int = 42) -> List[Dict[str, Any]]:
    """
    Synthetic bug-fixing dataset with simple lexical variations.
    """
    random.seed(seed)

    base_samples = [
        {
            "name": "mean_with_offset",
            "bug_type": "extra_offset",
            "buggy_code": """def mean(values):
    total = sum(values)
    count = len(values)
    return total / count - 1""",
            "fixed_code": """def mean(values):
    total = sum(values)
    count = len(values)
    return total / count""",
        },
        {
            "name": "normalize_with_shift",
            "bug_type": "extra_offset",
            "buggy_code": """def normalize(values):
    min_value = min(values)
    max_value = max(values)
    return [(x - min_value) / (max_value - min_value) + 1 for x in values]""",
            "fixed_code": """def normalize(values):
    min_value = min(values)
    max_value = max(values)
    return [(x - min_value) / (max_value - min_value) for x in values]""",
        },
        {
            "name": "accuracy_wrong_denominator",
            "bug_type": "wrong_denominator",
            "buggy_code": """def accuracy(predictions, labels):
    correct = sum(p == y for p, y in zip(predictions, labels))
    return correct / correct""",
            "fixed_code": """def accuracy(predictions, labels):
    correct = sum(p == y for p, y in zip(predictions, labels))
    return correct / len(labels)""",
        },
        {
            "name": "precision_wrong_denominator",
            "bug_type": "wrong_denominator",
            "buggy_code": """def precision(tp, fp):
    return tp / (tp + tp)""",
            "fixed_code": """def precision(tp, fp):
    return tp / (tp + fp)""",
        },
        {
            "name": "recall_wrong_denominator",
            "bug_type": "wrong_denominator",
            "buggy_code": """def recall(tp, fn):
    return tp / (tp + tp)""",
            "fixed_code": """def recall(tp, fn):
    return tp / (tp + fn)""",
        },
        {
            "name": "f1_wrong_formula",
            "bug_type": "wrong_formula",
            "buggy_code": """def f1_score(precision, recall):
    return precision * recall / (precision + recall)""",
            "fixed_code": """def f1_score(precision, recall):
    return 2 * precision * recall / (precision + recall)""",
        },
        {
            "name": "min_max_swapped",
            "bug_type": "swapped_min_max",
            "buggy_code": """def value_range(values):
    low = max(values)
    high = min(values)
    return high - low""",
            "fixed_code": """def value_range(values):
    low = min(values)
    high = max(values)
    return high - low""",
        },
        {
            "name": "clip_bounds_swapped",
            "bug_type": "swapped_bounds",
            "buggy_code": """def clip_value(x, lower, upper):
    if x < lower:
        return upper
    if x > upper:
        return lower
    return x""",
            "fixed_code": """def clip_value(x, lower, upper):
    if x < lower:
        return lower
    if x > upper:
        return upper
    return x""",
        },
        {
            "name": "filter_positive_wrong_sign",
            "bug_type": "wrong_condition",
            "buggy_code": """def keep_positive(values):
    result = []
    for x in values:
        if x < 0:
            result.append(x)
    return result""",
            "fixed_code": """def keep_positive(values):
    result = []
    for x in values:
        if x > 0:
            result.append(x)
    return result""",
        },
        {
            "name": "filter_long_strings_wrong_condition",
            "bug_type": "wrong_condition",
            "buggy_code": """def keep_long_strings(items, min_length):
    result = []
    for item in items:
        if len(item) < min_length:
            result.append(item)
    return result""",
            "fixed_code": """def keep_long_strings(items, min_length):
    result = []
    for item in items:
        if len(item) >= min_length:
            result.append(item)
    return result""",
        },
        {
            "name": "moving_average_wrong_window",
            "bug_type": "off_by_one_window",
            "buggy_code": """def moving_average(values, window):
    result = []
    for i in range(window, len(values)):
        chunk = values[i-window:i]
        result.append(sum(chunk) / window)
    return result""",
            "fixed_code": """def moving_average(values, window):
    result = []
    for i in range(window - 1, len(values)):
        chunk = values[i-window+1:i+1]
        result.append(sum(chunk) / window)
    return result""",
        },
        {
            "name": "train_test_split_wrong_slice",
            "bug_type": "wrong_slice",
            "buggy_code": """def split_train_test(items, train_size):
    split = int(len(items) * train_size)
    train = items[split:]
    test = items[:split]
    return train, test""",
            "fixed_code": """def split_train_test(items, train_size):
    split = int(len(items) * train_size)
    train = items[:split]
    test = items[split:]
    return train, test""",
        },
        {
            "name": "remove_duplicates_wrong_membership",
            "bug_type": "wrong_membership",
            "buggy_code": """def remove_duplicates(items):
    seen = set()
    result = []
    for item in items:
        if item in seen:
            result.append(item)
            seen.add(item)
    return result""",
            "fixed_code": """def remove_duplicates(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            result.append(item)
            seen.add(item)
    return result""",
        },
        {
            "name": "count_occurrences_wrong_increment",
            "bug_type": "wrong_increment",
            "buggy_code": """def count_occurrences(items):
    counts = {}
    for item in items:
        counts[item] = counts.get(item, 0)
    return counts""",
            "fixed_code": """def count_occurrences(items):
    counts = {}
    for item in items:
        counts[item] = counts.get(item, 0) + 1
    return counts""",
        },
        {
            "name": "softmax_missing_exp",
            "bug_type": "missing_transform",
            "buggy_code": """def softmax(values):
    total = sum(values)
    return [x / total for x in values]""",
            "fixed_code": """def softmax(values):
    import math
    exp_values = [math.exp(x) for x in values]
    total = sum(exp_values)
    return [x / total for x in exp_values]""",
        },
        {
            "name": "standardize_wrong_formula",
            "bug_type": "wrong_formula",
            "buggy_code": """def standardize(values):
    mean_value = sum(values) / len(values)
    variance = sum((x - mean_value) ** 2 for x in values) / len(values)
    return [(x + mean_value) / variance for x in values]""",
            "fixed_code": """def standardize(values):
    mean_value = sum(values) / len(values)
    variance = sum((x - mean_value) ** 2 for x in values) / len(values)
    std = variance ** 0.5
    return [(x - mean_value) / std for x in values]""",
        },
    ]

    replacements_pool = [
        {
            "values": "numbers",
            "items": "elements",
            "result": "output",
            "total": "sum_value",
            "count": "n",
            "x": "value",
            "item": "element",
        },
        {
            "values": "data",
            "items": "records",
            "result": "filtered",
            "total": "total_value",
            "count": "size",
            "x": "v",
            "item": "record",
        },
        {
            "values": "array",
            "items": "objects",
            "result": "answer",
            "total": "s",
            "count": "length",
            "x": "number",
            "item": "obj",
        },
    ]

    def apply_replacements(code: str, replacements: Dict[str, str]) -> str:
        for old, new in replacements.items():
            code = code.replace(old, new)
        return code

    samples = []

    for copy_id in range(copies):
        replacements = random.choice(replacements_pool)

        for sample in base_samples:
            function_suffix = f"_{copy_id}"
            buggy_code = sample["buggy_code"]
            fixed_code = sample["fixed_code"]

            buggy_code = apply_replacements(buggy_code, replacements)
            fixed_code = apply_replacements(fixed_code, replacements)

            first_line = buggy_code.split("\n")[0]
            function_name = first_line.split("def ")[1].split("(")[0]
            new_function_name = f"{function_name}{function_suffix}"

            buggy_code = buggy_code.replace(f"def {function_name}(", f"def {new_function_name}(")
            fixed_code = fixed_code.replace(f"def {function_name}(", f"def {new_function_name}(")

            new_sample = dict(sample)
            new_sample["id"] = f'{sample["name"]}_{copy_id}'
            new_sample["buggy_code"] = buggy_code
            new_sample["fixed_code"] = fixed_code

            samples.append(new_sample)

    random.shuffle(samples)
    return samples


synthetic_samples = build_harder_synthetic_samples(copies=10, seed=CONFIG["seed"])

print("Synthetic samples:", len(synthetic_samples))
print("Bug types:", sorted(set(x["bug_type"] for x in synthetic_samples)))

print("\nExample buggy code:")
print(synthetic_samples[0]["buggy_code"])

print("\nExample fixed code:")
print(synthetic_samples[0]["fixed_code"])

Synthetic samples: 160
Bug types: ['extra_offset', 'missing_transform', 'off_by_one_window', 'swapped_bounds', 'swapped_min_max', 'wrong_condition', 'wrong_denominator', 'wrong_formula', 'wrong_increment', 'wrong_membership', 'wrong_slice']

Example buggy code:
def accuracy_1(predictions, labels):
    correct = sum(p == y for p, y in zip(predictions, labels))
    return correct / correct

Example fixed code:
def accuracy_1(predictions, labels):
    correct = sum(p == y for p, y in zip(predictions, labels))
    return correct / len(labels)


#### SFT-формат: что именно подаём в модель


In [ ]:
def format_sft_text(sample: Dict[str, Any]) -> str:
    """Формат одной обучающей записи для causal LM fine-tuning."""
    return (
        "### Instruction:\n"
        "Fix the bug in this Python code. Return only corrected code.\n\n"
        "### Input:\n"
        f"{sample['buggy_code']}\n\n"
        "### Response:\n"
        f"{sample['fixed_code']}"
    )

print(format_sft_text(synthetic_samples[0]))


### Instruction:
Fix the bug in this Python code. Return only corrected code.

### Input:
def accuracy_1(predictions, labels):
    correct = sum(p == y for p, y in zip(predictions, labels))
    return correct / correct

### Response:
def accuracy_1(predictions, labels):
    correct = sum(p == y for p, y in zip(predictions, labels))
    return correct / len(labels)


In [ ]:
from datasets import Dataset


def make_datasets(samples: List[Dict[str, Any]], val_size: float = 0.2):
    random.shuffle(samples)
    split_idx = int(len(samples) * (1 - val_size))
    train_samples = samples[:split_idx]
    val_samples = samples[split_idx:]

    train_dataset = Dataset.from_list([
        {"text": format_sft_text(x), **x} for x in train_samples
    ])
    val_dataset = Dataset.from_list([
        {"text": format_sft_text(x), **x} for x in val_samples
    ])
    return train_dataset, val_dataset


train_dataset, val_dataset = make_datasets(synthetic_samples, val_size=0.2)
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))


Train: 128
Validation: 32


In [ ]:
def tokenize_function(batch):
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=CONFIG["max_length"],
    )

    # Для causal language modeling labels = input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names,
)

print(tokenized_train)
print(tokenized_val)


Map:   0%|          | 0/128 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 128
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 32
})


#### LoRA setup


In [ ]:
from peft import LoraConfig, get_peft_model


def prepare_lora_model(model):
    """Добавляет LoRA-адаптеры к base model."""
    if model is None:
        raise RuntimeError("Load base_model first")

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "v_proj"],
    )

    lora_model = get_peft_model(model, lora_config)
    lora_model.print_trainable_parameters()
    return lora_model


lora_model = prepare_lora_model(base_model)


trainable params: 1,572,864 || all params: 1,348,044,800 || trainable%: 0.1167


/home/dasha/.pyenv/versions/3.11.8/lib/python3.11/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/dasha/.pyenv/versions/3.11.8/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling


def build_trainer(lora_model, tokenized_train, tokenized_val):
    """Создаёт Trainer для LoRA fine-tuning."""
    training_args = TrainingArguments(
        output_dir=CONFIG["output_dir"],
        num_train_epochs=CONFIG["num_train_epochs"],
        per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        eval_strategy="epoch",       # если старая версия transformers ругается, заменить на evaluation_strategy
        save_strategy="epoch",
        learning_rate=CONFIG["learning_rate"],
        logging_steps=5,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    trainer = Trainer(
        model=lora_model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
    )

    return trainer


trainer = build_trainer(lora_model, tokenized_train, tokenized_val)
print("Trainer is ready")


The model is already on multiple devices. Skipping the move to device specified in `args`.


Trainer is ready


In [ ]:
if CONFIG["run_training"]:
    trainer.train()
    trainer.save_model(CONFIG["output_dir"])
    tokenizer.save_pretrained(CONFIG["output_dir"])
    print("Adapter saved to:", CONFIG["output_dir"])
else:
    print("Training skipped.")
    print("Set CONFIG['run_training'] = True to train LoRA adapter.")
    print("If adapter is already trained, continue to the next section.")


Epoch,Training Loss,Validation Loss
1,0.988200,0.832846
2,0.634300,0.524103
3,0.476700,0.439665


Adapter saved to: ./bugfix_lora_adapter


#### Загрузка fine-tuned adapter


In [ ]:
from peft import PeftModel


def load_finetuned_adapter(base_model, adapter_path: str):
    """Загружает обученный LoRA-адаптер поверх base model."""
    if base_model is None:
        raise RuntimeError("Load base_model first")

    if not os.path.exists(adapter_path):
        raise FileNotFoundError(
            f"Adapter not found: {adapter_path}. "
            "Run training first or set CONFIG['output_dir'] to existing adapter path."
        )

    ft_model = PeftModel.from_pretrained(base_model, adapter_path)
    print("Adapter loaded from:", adapter_path)
    return ft_model


ft_model = load_finetuned_adapter(base_model, CONFIG["output_dir"])


Adapter loaded from: ./bugfix_lora_adapter


/home/dasha/.pyenv/versions/3.11.8/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


#### Fine-tuned evaluation


In [ ]:
ft_rows = evaluate_local_model(
    samples=EVAL_SAMPLES,
    model=ft_model,
    title="ft",
)

ft_df = show_eval_table(ft_rows)


,name,bug_type,buggy_code,gold,ft_prediction,match
0,median_wrong_index,off_by_one_index,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,0
1,specificity_wrong_denominator,wrong_denominator,"def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...",0
2,negative_filter_wrong_sign,wrong_condition,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,0
3,min_max_scale_wrong_formula,wrong_formula,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,0
4,deduplicate_wrong_membership,wrong_membership,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,0
5,count_words_wrong_increment,wrong_increment,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,0


Exact match: 0.0


In [ ]:
ft_df["ft_prediction_clean"] = ft_df["ft_prediction"].apply(clean_prediction)

ft_df["exact_match_clean"] = ft_df.apply(
    lambda row: exact_match_score(row["ft_prediction_clean"], row["gold"]),
    axis=1,
)

ft_df["contains_gold_clean"] = ft_df.apply(
    lambda row: contains_gold_score(row["ft_prediction_clean"], row["gold"]),
    axis=1,
)

display(ft_df)

print("Raw exact match:", round(ft_df["match"].mean(), 3))
print("Clean exact match:", round(ft_df["exact_match_clean"].mean(), 3))
print("Clean contains gold:", round(ft_df["contains_gold_clean"].mean(), 3))

,name,bug_type,buggy_code,gold,ft_prediction,match,ft_prediction_clean,exact_match_clean,contains_gold_clean
0,median_wrong_index,off_by_one_index,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,0,def median_sorted(values):\n middle = len(v...,1,1
1,specificity_wrong_denominator,wrong_denominator,"def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...",0,"def specificity(tn, fp):\n return tn / (tn ...",1,1
2,negative_filter_wrong_sign,wrong_condition,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,0,def keep_negative(values):\n result = []\n ...,1,1
3,min_max_scale_wrong_formula,wrong_formula,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,0,def min_max_scale(values):\n low = min(valu...,1,1
4,deduplicate_wrong_membership,wrong_membership,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,0,def unique_values(values):\n seen = set()\n...,0,0
5,count_words_wrong_increment,wrong_increment,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,0,def count_words(words):\n counts = {}\n ...,1,1


Raw exact match: 0.0
Clean exact match: 0.833
Clean contains gold: 0.833


#### Baseline vs fine-tuned comparison


In [ ]:
def compare_rows(baseline_rows, ft_rows):
    comparison = []
    for base_row, ft_row in zip(baseline_rows, ft_rows):
        comparison.append({
            "name": base_row["name"],
            "bug_type": base_row["bug_type"],
            "buggy_code": base_row["buggy_code"],
            "gold": base_row["gold"],
            "base_prediction": base_row["base_prediction"],
            "ft_prediction": ft_row["ft_prediction"],
            "base_match": base_row["match"],
            "ft_match": ft_row["match"],
        })
    return pd.DataFrame(comparison)


comparison_df = compare_rows(baseline_rows, ft_rows)
display(comparison_df)

print("Base exact match:", round(comparison_df["base_match"].mean(), 3))
print("Fine-tuned exact match:", round(comparison_df["ft_match"].mean(), 3))


,name,bug_type,buggy_code,gold,base_prediction,ft_prediction,base_match,ft_match
0,median_wrong_index,off_by_one_index,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,0,0
1,specificity_wrong_denominator,wrong_denominator,"def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...",0,0
2,negative_filter_wrong_sign,wrong_condition,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,0,0
3,min_max_scale_wrong_formula,wrong_formula,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,0,0
4,deduplicate_wrong_membership,wrong_membership,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,0,0
5,count_words_wrong_increment,wrong_increment,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,0,0


Base exact match: 0.0
Fine-tuned exact match: 0.0


In [ ]:
def compare_clean_dfs(baseline_df, ft_df):
    comparison_df = pd.DataFrame({
        "name": baseline_df["name"],
        "bug_type": baseline_df["bug_type"],
        "buggy_code": baseline_df["buggy_code"],
        "gold": baseline_df["gold"],

        "base_prediction_raw": baseline_df["base_prediction"],
        "base_prediction_clean": baseline_df["base_prediction_clean"],

        "ft_prediction_raw": ft_df["ft_prediction"],
        "ft_prediction_clean": ft_df["ft_prediction_clean"],

        "base_exact_match_raw": baseline_df["match"],
        "ft_exact_match_raw": ft_df["match"],

        "base_exact_match_clean": baseline_df["exact_match_clean"],
        "ft_exact_match_clean": ft_df["exact_match_clean"],

        "base_contains_gold_clean": baseline_df["contains_gold_clean"],
        "ft_contains_gold_clean": ft_df["contains_gold_clean"],
    })

    return comparison_df


comparison_df = compare_clean_dfs(baseline_df, ft_df)
display(comparison_df)

print("Base raw exact match:", round(comparison_df["base_exact_match_raw"].mean(), 3))
print("Fine-tuned raw exact match:", round(comparison_df["ft_exact_match_raw"].mean(), 3))

print("Base clean exact match:", round(comparison_df["base_exact_match_clean"].mean(), 3))
print("Fine-tuned clean exact match:", round(comparison_df["ft_exact_match_clean"].mean(), 3))

print("Base clean contains gold:", round(comparison_df["base_contains_gold_clean"].mean(), 3))
print("Fine-tuned clean contains gold:", round(comparison_df["ft_contains_gold_clean"].mean(), 3))

,name,bug_type,buggy_code,gold,base_prediction_raw,base_prediction_clean,ft_prediction_raw,ft_prediction_clean,base_exact_match_raw,ft_exact_match_raw,base_exact_match_clean,ft_exact_match_clean,base_contains_gold_clean,ft_contains_gold_clean
0,median_wrong_index,off_by_one_index,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,def median_sorted(values):\n middle = len(v...,0,0,1,1,1,1
1,specificity_wrong_denominator,wrong_denominator,"def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...","def specificity(tn, fp):\n return tn / (tn ...",0,0,0,1,1,1
2,negative_filter_wrong_sign,wrong_condition,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,def keep_negative(values):\n result = []\n ...,0,0,0,1,1,1
3,min_max_scale_wrong_formula,wrong_formula,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,def min_max_scale(values):\n low = min(valu...,0,0,1,1,1,1
4,deduplicate_wrong_membership,wrong_membership,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,def unique_values(values):\n seen = set()\n...,0,0,0,0,0,0
5,count_words_wrong_increment,wrong_increment,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,def count_words(words):\n counts = {}\n ...,0,0,0,1,0,1


Base raw exact match: 0.0
Fine-tuned raw exact match: 0.0
Base clean exact match: 0.333
Fine-tuned clean exact match: 0.833
Base clean contains gold: 0.667
Fine-tuned clean contains gold: 0.833
